# 类继承关系
```mermaid
classDiagram
    class Records
    class Ranges
    
    %% 继承关系
    Records <|-- Ranges
```

# ranges_field_config 和 ranges_attach_field_config
参考 [records/decorators.ipynb](../records/decorators.ipynb) 中的 *关于例子的说明*：
- `range_dt`：表示 *范围* 的数据结构
- `ranges_field_config`：数据结构映射，包括每个字段的名称 `name`，显示标题 `title`，以及映射或取值范围 `mapping`
- `ranges_attach_field_config`：每个字段是否生成属性/可过滤的属性（根据取值）

```python
range_dt = np.dtype([
    ('id', np.int64),        # 唯一标识符：用于区分不同的范围
    ('col', np.int64),       # 列索引：指示该范围属于哪一列数据
    ('start_idx', np.int64), # 起始索引：范围开始的时间点索引
    ('end_idx', np.int64),   # 结束索引：范围结束的时间点索引
    ('status', np.int64)     # 状态标识：使用RangeStatus枚举值，标识范围的开放/关闭状态
], align=True)               # align=True确保字段在内存中对齐，提高访问性能和缓存效率
"""
```

```python
ranges_field_config = Config(
    dict(
        dtype=range_dt, # range_dt 包含了范围记录所需的所有字段定义
        
        settings=dict(
            id=dict(
                title='Range Id'
            ),
            
            # 索引字段配置 - 将idx字段重映射为end_idx
            idx=dict(
                name='end_idx'
            ),
            
            # 起始索引字段配置
            start_idx=dict(
                title='Start Timestamp',
                mapping='index'
            ),
            
            # 结束索引字段配置
            end_idx=dict(
                title='End Timestamp',
                mapping='index'
            ),
            
            # 状态字段配置
            status=dict(
                title='Status',         # 字段显示标题：状态
                mapping=RangeStatus     # 映射到RangeStatus枚举类
            )
        )
    ),
    readonly=True,
    as_attrs=False
)
```

```python
ranges_attach_field_config = Config(
    dict(
        status=dict(
            attach_filters=True 
        )
    ),
    readonly=True,
    as_attrs=False
)
```

# class Ranges(Records)
注意：
```python
@attach_fields(ranges_attach_field_config)
@override_field_config(ranges_field_config)
class Ranges(Records): ...
```

## `__init__`

```python
def __init__(self,
                wrapper: ArrayWrapper,
                records_arr: tp.RecordArray,
                ts: tp.Optional[tp.ArrayLike] = None,
                **kwargs) -> None:
    Records.__init__(
        self,
        wrapper,
        records_arr,
        ts=ts,
        **kwargs
    )
    self._ts = ts
```

## indexing_func
执行索引操作并返回新的 `Ranges` 实例。

```python
def indexing_func(self: RangesT, pd_indexing_func: tp.PandasIndexingFunc, **kwargs) -> RangesT:
    new_wrapper, new_records_arr, _, col_idxs = \
        Records.indexing_func_meta(self, pd_indexing_func, **kwargs)
    if self.ts is not None:
        new_ts = new_wrapper.wrap(self.ts.values[:, col_idxs], group_by=False)
    else:
        new_ts = None
    return self.replace(
        wrapper=new_wrapper,
        records_arr=new_records_arr,
        ts=new_ts
    )
```

## from_ts
从时间序列创建 `Ranges` 对象。

参数：
- `ts` (tp.ArrayLike)：输入的时间序列数据
    可以是Series、DataFrame或数组
- `gap_value` (tp.Optional[tp.Scalar], 可选)：间隔值
    如果为None，会根据数据类型自动选择
- `attach_ts` (bool, 可选)：是否附加原始时间序列，默认True
- `wrapper_kwargs` (tp.KwargsLike, 可选)：传递给ArrayWrapper的参数
- `**kwargs`：传递给Ranges构造函数的额外参数

返回：从时间序列创建的Ranges对象

```python
@classmethod
def from_ts(cls: tp.Type[RangesT],
            ts: tp.ArrayLike,
            gap_value: tp.Optional[tp.Scalar] = None,
            attach_ts: bool = True,
            wrapper_kwargs: tp.KwargsLike = None,
            **kwargs) -> RangesT:

    if wrapper_kwargs is None:
        wrapper_kwargs = {}

    ts_pd = to_pd_array(ts)
    ts_arr = to_2d_array(ts_pd)
    if gap_value is None:
        if np.issubdtype(ts_arr.dtype, np.bool_):
            # 布尔数据：False作为间隔
            gap_value = False
        elif np.issubdtype(ts_arr.dtype, np.integer):
            # 整数数据：-1作为间隔
            gap_value = -1
        else:
            # 其他数据类型：NaN作为间隔
            gap_value = np.nan
    # 扫描数组，找出所有连续的非间隔值序列
    records_arr = nb.find_ranges_nb(ts_arr, gap_value)
    wrapper = ArrayWrapper.from_obj(ts_pd, **wrapper_kwargs)
    return cls(wrapper, records_arr, ts=ts_pd if attach_ts else None, **kwargs)
```

### 例子

In [ ]:
import vectorbt as vbt
import pandas as pd
import numpy as np

# 示例1：从布尔序列创建范围
bool_ts = pd.Series([True, True, False, True, True, True, False])
ranges1 = vbt.Ranges.from_ts(bool_ts)
print("布尔范围:")
print(ranges1.records_readable)

In [ ]:
# 示例2：从整数序列创建范围
int_ts = pd.Series([1, 2, -1, 3, 4, 5, -1])
ranges2 = vbt.Ranges.from_ts(int_ts)
print("整数范围:")
print(ranges2.records_readable)

In [ ]:
# 示例3：从浮点序列创建范围
float_ts = pd.Series([1.5, 2.3, np.nan, 4.1, 5.2, np.nan, 6.7])
ranges3 = vbt.Ranges.from_ts(float_ts)
print("浮点范围:")
print(ranges3.records_readable)

In [ ]:
# 示例4：多列DataFrame
df = pd.DataFrame({
'A': [True, True, False, True, False],
'B': [False, True, True, False, True]
})
ranges4 = vbt.Ranges.from_ts(df)
print("多列范围:")
print(ranges4.records_readable)

In [ ]:
# 示例5：自定义间隔值
custom_ts = pd.Series([1, 2, 0, 3, 4, 0, 5])
ranges5 = vbt.Ranges.from_ts(custom_ts, gap_value=0)
print("自定义间隔范围:")
print(ranges5.records_readable)

In [ ]:
# 示例6：金融应用 - 识别上涨趋势
prices = pd.Series([100, 102, 104, 101, 103, 105, 102])
uptrend = prices.diff() > 0
uptrend_ranges = vbt.Ranges.from_ts(uptrend)
print("上涨趋势范围:")
print(uptrend_ranges.records_readable)

## to_mask
将范围转换为掩码：其中 `True` 表示在范围内，`False` 表示不在范围内。

```python
def to_mask(self, group_by: tp.GroupByLike = None, wrap_kwargs: tp.KwargsLike = None) -> tp.SeriesFrame:

    col_map = self.col_mapper.get_col_map(group_by=group_by)
    mask = nb.ranges_to_mask_nb(
        self.get_field_arr('start_idx'),
        self.get_field_arr('end_idx'),
        self.get_field_arr('status'),
        col_map,
        len(self.wrapper.index)
    )
    return self.wrapper.wrap(mask, group_by=group_by, **merge_dicts({}, wrap_kwargs))
```

### 例子

In [10]:
import vectorbt as vbt
import pandas as pd
import numpy as np

# 创建示例数据
ts = pd.Series([True, True, False, True, True, True, False])
ranges = vbt.Ranges.from_ts(ts)

# 转换为掩码
mask = ranges.to_mask()
print("范围掩码:")
print(mask)

范围掩码:
0     True
1     True
2    False
3     True
4     True
5     True
6    False
dtype: bool


In [11]:
# 多列数据示例
df = pd.DataFrame({
    'A': [True, True, False, True, False],
    'B': [False, True, True, False, True]
})
ranges_multi = vbt.Ranges.from_ts(df)
mask_multi = ranges_multi.to_mask()
print("多列掩码:")
print(mask_multi)

多列掩码:
       A      B
0   True  False
1   True   True
2  False   True
3   True  False
4  False   True


In [12]:
# 分组示例
grouped_mask = ranges_multi.to_mask(group_by=['Group1', 'Group1'])
print("分组掩码:")
print(grouped_mask)

分组掩码:
0    True
1    True
2    True
3    True
4    True
Name: Group1, dtype: bool


## duration
计算每个范围的持续时间。

```python
@cached_property
def duration(self) -> MappedArray:
    """Duration of each range (in raw format)."""
    duration = nb.range_duration_nb(
        self.get_field_arr('start_idx'),
        self.get_field_arr('end_idx'),
        self.get_field_arr('status')
    )
    return self.map_array(duration)
```

### 例子

In [ ]:
import vectorbt as vbt
import pandas as pd
import numpy as np

# 创建带时间频率的数据
ts_with_freq = pd.Series(
    [True, True, False, True, True, True, False],
    index=pd.date_range('2023-01-01', periods=7, freq='D')
)
ranges_with_freq = vbt.Ranges.from_ts(ts_with_freq)

durations = ranges_with_freq.duration
print("范围持续时间（索引单位）:")
print(durations.values)

## avg_duration
计算范围的平均持续时间。

```python
@cached_method
def avg_duration(self, group_by: tp.GroupByLike = None,
                    wrap_kwargs: tp.KwargsLike = None, **kwargs) -> tp.MaybeSeries:

    wrap_kwargs = merge_dicts(dict(to_timedelta=True, name_or_index='avg_duration'), wrap_kwargs)
    return self.duration.mean(group_by=group_by, wrap_kwargs=wrap_kwargs, **kwargs)
```

In [20]:
import vectorbt as vbt
import pandas as pd

# 创建带时间频率的范围数据
ts = pd.Series(
    [True, True, False, True, True, True, False, True],
    index=pd.date_range('2023-01-01', periods=8, freq='D')
)
ranges = vbt.Ranges.from_ts(ts)

# 计算平均持续时间
avg_dur = ranges.avg_duration()
print("平均持续时间:", avg_dur)

# 多列数据的平均持续时间
df = pd.DataFrame({
    'A': [True, True, False, True, False],
    'B': [False, True, True, False, True]
}, index=pd.date_range('2023-01-01', periods=5, freq='D'))

ranges_multi = vbt.Ranges.from_ts(df)
avg_dur_multi = ranges_multi.avg_duration()
print("多列平均持续时间:")
print(avg_dur_multi)

平均持续时间: 2 days 00:00:00
多列平均持续时间:
A   1 days 12:00:00
B   1 days 12:00:00
Name: avg_duration, dtype: timedelta64[s]
